# Geolocation Provider Search Demo
This notebook demonstrates Databricks geospatial capabilities for finding providers within a search radius of a customer location. It uses built-in spatial SQL functions (`ST_Point`, `ST_DistanceSpheroid`, `ST_DWithin`) to perform proximity searches on geocoded locations.

## Summary of Spatial Functions Used
| Function | Purpose |
| --- | --- |
| `ST_Point(lon, lat)` | Create a geometry point from longitude/latitude |
| `ST_DistanceSpheroid(geom1, geom2)` | Geodesic distance in meters (WGS84 ellipsoid) |
| **Tip**: For DBR 18.1+, use `ST_DWithin(geom1, geom2, distance)` in the JOIN predicate for optimized spatial index-based proximity joins |

## Step 1: Create Sample Data
Create temporary views with sample customer and provider locations (realistic US coordinates).

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "schema", "Schema")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

#spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"USE CATALOG {catalog}")

#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
%sql
CREATE OR REPLACE TABLE customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson',   '123 Main St, Denver, CO',          39.7392, -104.9903),
  (2, 'Bob Smith',       '456 Oak Ave, Boulder, CO',         40.0150, -105.2705),
  (3, 'Carol Davis',     '789 Pine Rd, Colorado Springs, CO',38.8339, -104.8214),
  (4, 'Dan Wilson',      '321 Elm St, Fort Collins, CO',     40.5853, -105.0844),
  (5, 'Eva Martinez',    '654 Maple Dr, Aurora, CO',         39.7294, -104.8319)
AS customers(customer_id, customer_name, address, lat, lon);

SELECT * FROM customers;

In [0]:
%sql
CREATE OR REPLACE TABLE providers AS
SELECT * FROM VALUES
  -- Primary Care (8 providers)
  (101, 'Summit Health Clinic',         'Denver, CO',            39.7486, -104.9997, 'Primary Care'),
  (105, 'Pikes Peak Family Medicine',   'Colorado Springs, CO',  38.8339, -104.8214, 'Primary Care'),
  (108, 'Prairie Health Partners',      'Greeley, CO',           40.4233, -104.7091, 'Primary Care'),
  (112, 'Columbine Wellness Center',    'Littleton, CO',         39.6133, -105.0166, 'Primary Care'),
  (113, 'Highlands Family Practice',    'Highlands Ranch, CO',   39.5519, -104.9669, 'Primary Care'),
  (114, 'Cherry Creek Medical Group',   'Denver, CO',            39.7171, -104.9422, 'Primary Care'),
  (115, 'Arvada Family Health',         'Arvada, CO',            39.8028, -105.0875, 'Primary Care'),
  (116, 'Parker Primary Care',          'Parker, CO',            39.5186, -104.7614, 'Primary Care'),

  -- Dental (6 providers)
  (102, 'Rocky Mountain Dental',        'Lakewood, CO',          39.7047, -105.0814, 'Dental'),
  (117, 'Smile Studios Denver',         'Denver, CO',            39.7500, -104.9847, 'Dental'),
  (118, 'Boulder Dental Arts',          'Boulder, CO',           40.0189, -105.2747, 'Dental'),
  (119, 'Springs Dental Care',          'Colorado Springs, CO',  38.8608, -104.7902, 'Dental'),
  (120, 'Broomfield Family Dentistry',  'Broomfield, CO',        39.9388, -105.0528, 'Dental'),
  (121, 'Thornton Smiles',              'Thornton, CO',          39.8681, -104.9719, 'Dental'),

  -- Vision (6 providers)
  (103, 'Peak Vision Center',           'Boulder, CO',           40.0274, -105.2519, 'Vision'),
  (109, 'Flatiron Eye Associates',      'Broomfield, CO',        39.9205, -105.0867, 'Vision'),
  (122, 'Mile High Eye Care',           'Denver, CO',            39.7392, -105.0101, 'Vision'),
  (123, 'Crystal Clear Optometry',      'Aurora, CO',            39.7088, -104.8072, 'Vision'),
  (124, 'Mountain Sight Clinic',        'Golden, CO',            39.7636, -105.2094, 'Vision'),
  (125, 'Springs Eye Center',           'Colorado Springs, CO',  38.8497, -104.8497, 'Vision'),

  -- Orthopedics (5 providers)
  (104, 'Front Range Orthopedics',      'Longmont, CO',          40.1672, -105.1019, 'Orthopedics'),
  (126, 'Denver Bone & Joint',          'Denver, CO',            39.7278, -104.9731, 'Orthopedics'),
  (127, 'Colorado Sports Medicine',     'Boulder, CO',           40.0058, -105.2512, 'Orthopedics'),
  (128, 'Peak Orthopedic Specialists',  'Colorado Springs, CO',  38.8675, -104.7606, 'Orthopedics'),
  (129, 'Lakewood Ortho Center',        'Lakewood, CO',          39.7108, -105.1214, 'Orthopedics'),

  -- Urgent Care (5 providers)
  (106, 'Mile High Urgent Care',        'Centennial, CO',        39.5792, -104.8769, 'Urgent Care'),
  (130, 'QuickCare Denver',             'Denver, CO',            39.7619, -104.9811, 'Urgent Care'),
  (131, 'Boulder Immediate Care',       'Boulder, CO',           39.9997, -105.2581, 'Urgent Care'),
  (132, 'Springs Urgent Med',           'Colorado Springs, CO',  38.8781, -104.8317, 'Urgent Care'),
  (133, 'Aurora Express Clinic',        'Aurora, CO',            39.7294, -104.7953, 'Urgent Care'),

  -- Physical Therapy (5 providers)
  (107, 'Alpine Physical Therapy',      'Golden, CO',            39.7555, -105.2211, 'Physical Therapy'),
  (134, 'Restore PT Denver',            'Denver, CO',            39.7425, -105.0053, 'Physical Therapy'),
  (135, 'Active Recovery Boulder',      'Boulder, CO',           40.0331, -105.2597, 'Physical Therapy'),
  (136, 'Peak Performance PT',          'Colorado Springs, CO',  38.8200, -104.8406, 'Physical Therapy'),
  (137, 'FitLife Physical Therapy',     'Littleton, CO',         39.6028, -105.0331, 'Physical Therapy'),

  -- Pediatrics (5 providers)
  (110, 'Canyon Creek Pediatrics',      'Castle Rock, CO',       39.3722, -104.8561, 'Pediatrics'),
  (138, 'Little Stars Pediatrics',      'Denver, CO',            39.7331, -104.9669, 'Pediatrics'),
  (139, 'Growing Well Kids Clinic',     'Boulder, CO',           40.0147, -105.2408, 'Pediatrics'),
  (140, 'Happy Hearts Pediatrics',      'Aurora, CO',            39.6956, -104.8364, 'Pediatrics'),
  (141, 'Mountain Kids Health',         'Fort Collins, CO',      40.5742, -105.0653, 'Pediatrics'),

  -- Cardiology (5 providers)
  (111, 'Mountain View Cardiology',     'Westminster, CO',       39.8367, -105.0372, 'Cardiology'),
  (142, 'Heart of the Rockies',         'Denver, CO',            39.7481, -104.9864, 'Cardiology'),
  (143, 'Colorado Cardiac Associates',  'Lakewood, CO',          39.7256, -105.0906, 'Cardiology'),
  (144, 'Boulder Heart Center',         'Boulder, CO',           40.0186, -105.2633, 'Cardiology'),
  (145, 'Springs Cardiology Group',     'Colorado Springs, CO',  38.8522, -104.8108, 'Cardiology'),

  -- Dermatology (4 providers)
  (146, 'Clear Skin Dermatology',       'Denver, CO',            39.7511, -104.9994, 'Dermatology'),
  (147, 'Rocky Mountain Skin Care',     'Boulder, CO',           40.0222, -105.2636, 'Dermatology'),
  (148, 'SunSafe Dermatology',          'Colorado Springs, CO',  38.8456, -104.8139, 'Dermatology'),
  (149, 'Centennial Derm Associates',   'Centennial, CO',        39.5847, -104.8761, 'Dermatology'),

  -- Mental Health (4 providers)
  (150, 'Mindful Wellness Center',      'Denver, CO',            39.7397, -104.9847, 'Mental Health'),
  (151, 'Serenity Counseling',          'Boulder, CO',           40.0142, -105.2522, 'Mental Health'),
  (152, 'Thrive Mental Health',         'Aurora, CO',            39.7253, -104.8178, 'Mental Health'),
  (153, 'Mountain Peace Therapy',       'Fort Collins, CO',      40.5614, -105.0775, 'Mental Health')
AS providers(provider_id, provider_name, city, lat, lon, specialty);

SELECT specialty, COUNT(*) AS provider_count FROM providers GROUP BY specialty ORDER BY provider_count DESC;

## Step 2: Validate Geocoding
Ensure all customer and provider records have valid latitude and longitude values.

In [0]:
%sql
SELECT 'Customers' AS source,
       COUNT(*) AS total_records,
       COUNT_IF(lat IS NOT NULL AND lon IS NOT NULL) AS geocoded,
       COUNT_IF(lat IS NULL OR lon IS NULL) AS missing_geocode,
       ROUND(COUNT_IF(lat IS NOT NULL AND lon IS NOT NULL) * 100.0 / COUNT(*), 1) AS pct_geocoded
FROM customers
UNION ALL
SELECT 'Providers',
       COUNT(*),
       COUNT_IF(lat IS NOT NULL AND lon IS NOT NULL),
       COUNT_IF(lat IS NULL OR lon IS NULL),
       ROUND(COUNT_IF(lat IS NOT NULL AND lon IS NOT NULL) * 100.0 / COUNT(*), 1)
FROM providers;

## Step 3: Provider Proximity Search
Find all providers within a configurable search radius of each customer. Uses `ST_Point` to create geometry objects and `ST_DistanceSpheroid` for accurate geodesic distance calculation on the WGS84 ellipsoid.

In [0]:
%sql
DECLARE OR REPLACE radius_km DOUBLE = 100;
DECLARE OR REPLACE cust_name = 'Alice Johnson';
DECLARE OR REPLACE specialty_value = 'Primary Care';

WITH c AS (
  SELECT customer_id,
         customer_name,
         address,
         lat  AS cust_lat,
         lon  AS cust_lon,
         ST_Point(lon, lat) AS cust_geom
  FROM customers
),
p AS (
  SELECT provider_id,
         provider_name,
         city,
         specialty,
         lat  AS prov_lat,
         lon  AS prov_lon,
         ST_Point(lon, lat) AS prov_geom
  FROM providers
)
SELECT
  c.customer_id,
  c.customer_name,
  c.address        AS customer_address,
  p.provider_id,
  p.provider_name,
  p.specialty,
  p.city           AS provider_city,
  c.cust_lat,
  c.cust_lon,
  p.prov_lat,
  p.prov_lon,
  ROUND(ST_DistanceSpheroid(c.cust_geom, p.prov_geom) / 1000.0, 2) AS distance_km
FROM c
JOIN p
  ON ST_DistanceSpheroid(c.cust_geom, p.prov_geom) <= radius_km * 1000
WHERE 
  customer_name = cust_name 
  AND p.specialty = specialty_value
ORDER BY distance_km